# Task C — step 2: hard gate, P_θ vs Heston-P

Thresholds are pre-registered in `taskc/DECISIONS.md` §7 and hard-coded in `taskc/gate.py` (`GateThresholds`). This notebook loads draw A from step 1, compares it with an **independent** Heston-P sample (seed 20260922), prints every check, and plots the diagnostics. It changes nothing.

**Any failure ⇒ do not proceed to the dual.** Escalation ladder is in DECISIONS.md §7; each rung is logged in §9 and the gate is re-run on a fresh draw A.

Cell 0 is the Colab clone/checkout cell; locally, skip it.

In [ ]:
%cd /content
%rm -rf ddpm_option_pricing
!git clone https://github.com/nilay47/ddpm_option_pricing.git
%cd ddpm_option_pricing
!git fetch --all
!git checkout v2_code
!git status

In [ ]:
import os, sys, json
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
import taskc
from taskc.config import CFG
from taskc.data import reference_paths
from taskc.ptheta import load_checkpoint, load_draw
from taskc.gate import run_gate, print_report, summary_dict, THRESH, sv_stats
from config import q_params            # taskb
import evaluation as ev                 # taskb

from dataclasses import replace
RUN_TAG = "rung0"          # which rung's artifacts to gate
RUN = replace(CFG, artifact_dir=CFG.run_dir(RUN_TAG))
CKPT = os.path.join(RUN.artifact_dir, RUN.ckpt_name)
model, std, cfg_dict, extra = load_checkpoint(CKPT)
A = load_draw(RUN, "A")
print("checkpoint:", extra)
print(f"draw A: seed {A.seed}, n={A.z.shape[0]:,}, rejected {A.n_rejected} of {A.n_drawn:,}")
assert A.z.shape[0] == CFG.draws["A"].n, "draw A is not full size -- the gate needs the pre-registered N"
print("thresholds:", THRESH)

## Run the gate

In [ ]:
ref = reference_paths(CFG)
report = run_gate(A.z, std, CFG, THRESH, ref=ref)
print_report(report, columns=True)
with open(os.path.join(RUN.artifact_dir, "gate_report_nb.json"), "w") as f:
    json.dump(summary_dict(report), f, indent=2)

## Diagnostics

Per-column mean error and sd ratio with the pre-registered bounds; implied-vol smile of P_θ vs Heston-P at 21d and 10d (using r for both — a *P-smile* diagnostic, not a price); realized-variance dispersion.

In [ ]:
r = report
van = r.kinds == "vanilla"
x = np.arange(len(r.names))
fig, ax = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
ax[0].bar(x[~van], r.dmean_sd[~van], color="tab:blue", label="martingale cols (bound 0.05)")
ax[0].bar(x[van], r.dmean_sd[van], color="tab:orange", label="vanilla cols (bound 0.02)")
ax[0].axhline(THRESH.g1_mean, c="tab:blue", ls="--", lw=1); ax[0].axhline(THRESH.g1_tail, c="tab:orange", ls="--", lw=1)
ax[0].axhline(r.mc_floor_sd, c="k", ls=":", lw=1, label="MC floor"); ax[0].set_ylabel("|E_A - E_ref| / sd_ref"); ax[0].legend()
ax[1].bar(x[~van], r.sd_ratio[~van] - 1, color="tab:blue"); ax[1].bar(x[van], r.sd_ratio[van] - 1, color="tab:orange")
for b, c in ((THRESH.g2_mart, "tab:blue"), (THRESH.g2_van, "tab:orange")):
    ax[1].axhline(b, c=c, ls="--", lw=1); ax[1].axhline(-b, c=c, ls="--", lw=1)
ax[1].set_ylabel("sd_A / sd_ref - 1"); ax[1].set_xticks(x); ax[1].set_xticklabels(r.names, rotation=90, fontsize=6)
plt.tight_layout(); plt.show()

In [ ]:
pA = std.to_paths(A.z)
strikes = np.arange(85, 116, 2.5)
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for axi, step in zip(ax, (21, 10)):
    _, ivA = ev.smile(pA, strikes, step); _, ivR = ev.smile(ref, strikes, step)
    axi.plot(strikes, ivR * 100, marker="o", label="Heston-P (MC)"); axi.plot(strikes, ivA * 100, marker="x", label="P_theta A")
    axi.set_title(f"P-smile at {step}d (IV in %, r used for both)"); axi.set_xlabel("K"); axi.legend(); axi.grid(alpha=.3)
plt.tight_layout(); plt.show()

YA, YR = pA.returns(), ref.returns()
rvA, rvR = (YA**2).mean(1) / CFG.dt, (YR**2).mean(1) / CFG.dt
plt.figure(figsize=(6, 3.5)); bins = np.linspace(0, 0.15, 100)
plt.hist(rvR, bins=bins, density=True, alpha=.5, label="Heston-P"); plt.hist(rvA, bins=bins, density=True, alpha=.5, label="P_theta A")
plt.xlabel("RV_21 (annualised)"); plt.legend(); plt.title(f"realized-variance dispersion: CV A {r.sv_A['cv_rv']:.3f} vs ref {r.sv_ref['cv_rv']:.3f}"); plt.show()

# leverage profile by lag (diagnostic)
lags = range(1, 8)
levA = [np.corrcoef(YA[:, :-k].ravel(), (YA[:, k:]**2).ravel())[0, 1] for k in lags]
levR = [np.corrcoef(YR[:, :-k].ravel(), (YR[:, k:]**2).ravel())[0, 1] for k in lags]
plt.figure(figsize=(5, 3)); plt.plot(list(lags), levR, marker="o", label="Heston-P"); plt.plot(list(lags), levA, marker="x", label="P_theta A")
plt.axhline(0, c="k", lw=.8); plt.xlabel("lag k"); plt.ylabel("corr(Y_s, Y_{s+k}^2)"); plt.legend(); plt.grid(alpha=.3); plt.show()

## Verdict

`report.passed` is the gate. If it is `False`, stop: log the failing checks and the chosen rung of the escalation ladder in `taskc/DECISIONS.md` §9, retrain, regenerate **all three** draws, and re-run this notebook. Do not build constraints on these draws.

In [ ]:
print("GATE PASSED" if report.passed else "GATE FAILED -- do not proceed to the dual")
for c in report.checks:
    if not c.passed and c.id != "G1-info":
        print("  FAIL", c.id, f"{c.value:.4f}", c.bound, "|", c.detail)